<a href="https://colab.research.google.com/github/quiltroconiphone/logica_progamacion_parte1/blob/master/S2_Titanic_SparkSQL_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso Práctico — Análisis del Titanic con Apache Spark
### MCDI502 · Unidad 2 · Actividad práctica individual (formativa)

**Autor:** Roberto Sánchez Saldivia
**Entorno:** Google Colab + PySpark
**Dataset:** `Titanic.csv` — 891 pasajeros, 12 columnas.

Colab no trae PySpark instalado, así que la primera celda lo instala. Java sí viene incluido,
por eso no hay que configurar `JAVA_HOME`.

**Regla de la entrega:** las Partes 2, 3, 4 y 6 van **todas en Spark SQL**.

---
## PARTE 1 · Configuración y exploración inicial

### 1.1 Configuración del entorno

In [1]:
# Colab no trae PySpark. -q = instalación silenciosa (sin el muro de texto del pip).
# Tarda ~40 segundos la primera vez. Si reinicias el entorno hay que volver a correrla.
!pip install -q pyspark

import pyspark
print("PySpark", pyspark.__version__, "instalado")

PySpark 4.0.5 instalado


In [2]:
from pyspark.sql import SparkSession

# SparkSession es la puerta de entrada. local[*] = usa todos los núcleos de esta máquina.
spark = (SparkSession.builder
         .appName("TitanicAnalysis")
         .master("local[*]")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")   # silencia los WARN de hadoop

print("Spark", spark.version)
print("Núcleos (trabajadores):", spark.sparkContext.defaultParallelism)

Spark 4.0.5
Núcleos (trabajadores): 2


In [3]:
# Subir el archivo Titanic.csv desde el computador.
# Al ejecutar esta celda aparece un botón "Elegir archivos": busca Titanic.csv y ábrelo.
from google.colab import files
import os

if not os.path.exists("Titanic.csv"):
    files.upload()

print("Archivo listo:", os.path.exists("Titanic.csv"))

Saving Titanic.csv to Titanic.csv
Archivo listo: True


In [4]:
# header=True usa la primera fila como nombres de columna.
# inferSchema=True hace que Spark adivine los tipos (lee el archivo dos veces, pero acá es chico).
df = spark.read.csv("Titanic.csv", header=True, inferSchema=True)

print("Filas:", df.count(), "| Columnas:", len(df.columns))

Filas: 891 | Columnas: 12


### 1.2 Exploración básica

`printSchema()` muestra las columnas con su tipo. Es el equivalente a mirar los íconos
123 / ABC en Power Query: si una edad apareciera como `string`, no podría promediarla.

In [5]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [6]:
# Primeras 10 filas. truncate=False para que no corte los nombres largos.
df.show(10, truncate=False)

+-----------+--------+------+---------------------------------------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|Name                                               |Sex   |Age |SibSp|Parch|Ticket          |Fare   |Cabin|Embarked|
+-----------+--------+------+---------------------------------------------------+------+----+-----+-----+----------------+-------+-----+--------+
|1          |0       |3     |Braund, Mr. Owen Harris                            |male  |22.0|1    |0    |A/5 21171       |7.25   |NULL |S       |
|2          |1       |1     |Cumings, Mrs. John Bradley (Florence Briggs Thayer)|female|38.0|1    |0    |PC 17599        |71.2833|C85  |C       |
|3          |1       |3     |Heikkinen, Miss. Laina                             |female|26.0|0    |0    |STON/O2. 3101282|7.925  |NULL |S       |
|4          |1       |1     |Futrelle, Mrs. Jacques Heath (Lily May Peel)       |female|35.0|1    |0    |113803          |53

In [7]:
# Conteo total de pasajeros
total = df.count()
print("Total de pasajeros:", total)

Total de pasajeros: 891


In [8]:
# Estadísticas descriptivas de las columnas numéricas.
# Ojo: describe() ignora los nulos, por eso 'count' de Age es menor que 891.
df.describe("Survived", "Pclass", "Age", "SibSp", "Parch", "Fare").show()

+-------+-------------------+------------------+------------------+------------------+-------------------+-----------------+
|summary|           Survived|            Pclass|               Age|             SibSp|              Parch|             Fare|
+-------+-------------------+------------------+------------------+------------------+-------------------+-----------------+
|  count|                891|               891|               714|               891|                891|              891|
|   mean| 0.3838383838383838| 2.308641975308642| 29.69911764705882|0.5230078563411896|0.38159371492704824| 32.2042079685746|
| stddev|0.48659245426485753|0.8360712409770491|14.526497332334035|1.1027434322934315| 0.8060572211299488|49.69342859718089|
|    min|                  0|                 1|              0.42|                 0|                  0|              0.0|
|    max|                  1|                 3|              80.0|                 8|                  6|         512.3292|


**Lectura de `describe()`:** `Age` tiene `count` menor que 891. Esa diferencia son los nulos
y es justo lo que la Parte 5.1 pide identificar.

---
## PARTE 2 · Spark SQL — consultas básicas

Primero registro el DataFrame como **vista temporal**. Una vista no copia los datos:
es solo un nombre para poder consultarlo con SQL. Dura mientras dure la sesión de Spark.

In [9]:
df.createOrReplaceTempView("Titanic")
print("Vista temporal 'Titanic' registrada")
spark.sql("SHOW TABLES").show()

Vista temporal 'Titanic' registrada
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         |  titanic|       true|
+---------+---------+-----------+



### 2.1 ¿Cuántos pasajeros sobrevivieron y cuántos no?

In [10]:
# CASE WHEN traduce 0/1 a texto legible. GROUP BY agrupa; COUNT(*) cuenta cada grupo.
spark.sql("""
    SELECT
        CASE WHEN Survived = 1 THEN 'Sobrevivió' ELSE 'No sobrevivió' END AS Estado,
        COUNT(*) AS Cantidad
    FROM Titanic
    GROUP BY Survived
    ORDER BY Survived DESC
""").show()

+-------------+--------+
|       Estado|Cantidad|
+-------------+--------+
|   Sobrevivió|     342|
|No sobrevivió|     549|
+-------------+--------+



### 2.2 Distribución de pasajeros por clase

In [11]:
spark.sql("""
    SELECT Pclass, COUNT(*) AS Cantidad
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+--------+
|Pclass|Cantidad|
+------+--------+
|     1|     216|
|     2|     184|
|     3|     491|
+------+--------+



### 2.3 Distribución de pasajeros por sexo

In [12]:
spark.sql("""
    SELECT Sex, COUNT(*) AS Cantidad
    FROM Titanic
    GROUP BY Sex
    ORDER BY Cantidad DESC
""").show()

+------+--------+
|   Sex|Cantidad|
+------+--------+
|  male|     577|
|female|     314|
+------+--------+



### 2.4 Distribución por puerto de embarque

C = Cherbourg · Q = Queenstown · S = Southampton. El `null` son pasajeros sin dato de puerto.

In [13]:
spark.sql("""
    SELECT
        Embarked,
        CASE Embarked
            WHEN 'C' THEN 'Cherbourg'
            WHEN 'Q' THEN 'Queenstown'
            WHEN 'S' THEN 'Southampton'
            ELSE 'Sin dato'
        END AS Puerto,
        COUNT(*) AS Cantidad
    FROM Titanic
    GROUP BY Embarked
    ORDER BY Cantidad DESC
""").show()

+--------+-----------+--------+
|Embarked|     Puerto|Cantidad|
+--------+-----------+--------+
|       S|Southampton|     644|
|       C|  Cherbourg|     168|
|       Q| Queenstown|      77|
|    NULL|   Sin dato|       2|
+--------+-----------+--------+



### 2.5 Edad promedio, mínima y máxima

In [14]:
# ROUND(,2) deja 2 decimales. Las funciones de agregación ignoran los nulos de Age.
spark.sql("""
    SELECT
        ROUND(AVG(Age), 2) AS Edad_Promedio,
        MIN(Age)           AS Edad_Minima,
        MAX(Age)           AS Edad_Maxima,
        COUNT(Age)         AS Con_Edad_Registrada
    FROM Titanic
""").show()

+-------------+-----------+-----------+-------------------+
|Edad_Promedio|Edad_Minima|Edad_Maxima|Con_Edad_Registrada|
+-------------+-----------+-----------+-------------------+
|         29.7|       0.42|       80.0|                714|
+-------------+-----------+-----------+-------------------+



### 2.6 Tarifa promedio, mínima y máxima

In [15]:
spark.sql("""
    SELECT
        ROUND(AVG(Fare), 2) AS Tarifa_Promedio,
        MIN(Fare)           AS Tarifa_Minima,
        MAX(Fare)           AS Tarifa_Maxima
    FROM Titanic
""").show()

+---------------+-------------+-------------+
|Tarifa_Promedio|Tarifa_Minima|Tarifa_Maxima|
+---------------+-------------+-------------+
|           32.2|          0.0|     512.3292|
+---------------+-------------+-------------+



La tarifa mínima es **0**. No es un error de carga: hay pasajeros que viajaban sin pagar
(empleados de la naviera, tripulación invitada). Lo dejo como está porque 0 acá sí significa
"pagó cero", no "sin dato".

---
## PARTE 3 · Spark SQL — agregaciones por grupos

### 3.1 Tasa de supervivencia por clase

In [16]:
# SUM(Survived) funciona como contador porque Survived es 0 o 1:
# sumar unos equivale a contar sobrevivientes.
spark.sql("""
    SELECT
        Pclass,
        SUM(Survived)                                   AS Sobrevivientes,
        COUNT(*)                                        AS Total,
        ROUND(100.0 * SUM(Survived) / COUNT(*), 2)      AS Pct_Supervivencia
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+--------------+-----+-----------------+
|Pclass|Sobrevivientes|Total|Pct_Supervivencia|
+------+--------------+-----+-----------------+
|     1|           136|  216|            62.96|
|     2|            87|  184|            47.28|
|     3|           119|  491|            24.24|
+------+--------------+-----+-----------------+



### 3.2 Tasa de supervivencia por sexo

In [17]:
spark.sql("""
    SELECT
        Sex,
        SUM(Survived)                              AS Sobrevivientes,
        COUNT(*)                                   AS Total,
        ROUND(100.0 * SUM(Survived) / COUNT(*), 2) AS Pct_Supervivencia
    FROM Titanic
    GROUP BY Sex
    ORDER BY Pct_Supervivencia DESC
""").show()

+------+--------------+-----+-----------------+
|   Sex|Sobrevivientes|Total|Pct_Supervivencia|
+------+--------------+-----+-----------------+
|female|           233|  314|            74.20|
|  male|           109|  577|            18.89|
+------+--------------+-----+-----------------+



### 3.3 Edad promedio por clase y estado de supervivencia

In [18]:
spark.sql("""
    SELECT
        Pclass,
        CASE WHEN Survived = 1 THEN 'Sobrevivió' ELSE 'No sobrevivió' END AS Estado,
        ROUND(AVG(Age), 2) AS Edad_Promedio,
        COUNT(*)           AS Pasajeros
    FROM Titanic
    GROUP BY Pclass, Survived
    ORDER BY Pclass, Survived DESC
""").show()

+------+-------------+-------------+---------+
|Pclass|       Estado|Edad_Promedio|Pasajeros|
+------+-------------+-------------+---------+
|     1|   Sobrevivió|        35.37|      136|
|     1|No sobrevivió|         43.7|       80|
|     2|   Sobrevivió|         25.9|       87|
|     2|No sobrevivió|        33.54|       97|
|     3|   Sobrevivió|        20.65|      119|
|     3|No sobrevivió|        26.56|      372|
+------+-------------+-------------+---------+



### 3.4 Tarifa promedio por clase

In [19]:
spark.sql("""
    SELECT
        Pclass,
        ROUND(AVG(Fare), 2) AS Tarifa_Promedio,
        COUNT(*)            AS Pasajeros
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+---------------+---------+
|Pclass|Tarifa_Promedio|Pasajeros|
+------+---------------+---------+
|     1|          84.15|      216|
|     2|          20.66|      184|
|     3|          13.68|      491|
+------+---------------+---------+



### 3.5 Distribución de supervivencia por puerto de embarque

In [20]:
# SUM(CASE WHEN ...) es un conteo condicional: cuenta solo las filas que cumplen.
spark.sql("""
    SELECT
        Embarked,
        SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) AS Sobrevivientes,
        SUM(CASE WHEN Survived = 0 THEN 1 ELSE 0 END) AS No_Sobrevivientes,
        COUNT(*)                                      AS Total,
        ROUND(100.0 * SUM(Survived) / COUNT(*), 2)    AS Pct_Supervivencia
    FROM Titanic
    GROUP BY Embarked
    ORDER BY Total DESC
""").show()

+--------+--------------+-----------------+-----+-----------------+
|Embarked|Sobrevivientes|No_Sobrevivientes|Total|Pct_Supervivencia|
+--------+--------------+-----------------+-----+-----------------+
|       S|           217|              427|  644|            33.70|
|       C|            93|               75|  168|            55.36|
|       Q|            30|               47|   77|            38.96|
|    NULL|             2|                0|    2|           100.00|
+--------+--------------+-----------------+-----+-----------------+



### 3.6 Número promedio de familiares a bordo por clase

`Familiares = SibSp + Parch`: hermanos/cónyuges más padres/hijos.

In [21]:
spark.sql("""
    SELECT
        Pclass,
        ROUND(AVG(SibSp + Parch), 2) AS Familiares_Promedio,
        MAX(SibSp + Parch)           AS Familiares_Maximo
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+-------------------+-----------------+
|Pclass|Familiares_Promedio|Familiares_Maximo|
+------+-------------------+-----------------+
|     1|               0.77|                5|
|     2|               0.78|                5|
|     3|               1.01|               10|
+------+-------------------+-----------------+



---
## PARTE 4 · Spark SQL — consultas avanzadas

### 4.1 Solo los pasajeros que sobrevivieron

In [22]:
sobrevivientes = spark.sql("SELECT * FROM Titanic WHERE Survived = 1")
print("Sobrevivientes:", sobrevivientes.count())
sobrevivientes.select("PassengerId", "Name", "Sex", "Age", "Pclass").show(5, truncate=False)

Sobrevivientes: 342
+-----------+---------------------------------------------------+------+----+------+
|PassengerId|Name                                               |Sex   |Age |Pclass|
+-----------+---------------------------------------------------+------+----+------+
|2          |Cumings, Mrs. John Bradley (Florence Briggs Thayer)|female|38.0|1     |
|3          |Heikkinen, Miss. Laina                             |female|26.0|3     |
|4          |Futrelle, Mrs. Jacques Heath (Lily May Peel)       |female|35.0|1     |
|9          |Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  |female|27.0|3     |
|10         |Nasser, Mrs. Nicholas (Adele Achem)                |female|14.0|2     |
+-----------+---------------------------------------------------+------+----+------+
only showing top 5 rows


### 4.2 Solo pasajeras de sexo femenino

In [23]:
mujeres = spark.sql("SELECT * FROM Titanic WHERE Sex = 'female'")
print("Pasajeras:", mujeres.count())
mujeres.select("PassengerId", "Name", "Age", "Pclass", "Survived").show(5, truncate=False)

Pasajeras: 314
+-----------+---------------------------------------------------+----+------+--------+
|PassengerId|Name                                               |Age |Pclass|Survived|
+-----------+---------------------------------------------------+----+------+--------+
|2          |Cumings, Mrs. John Bradley (Florence Briggs Thayer)|38.0|1     |1       |
|3          |Heikkinen, Miss. Laina                             |26.0|3     |1       |
|4          |Futrelle, Mrs. Jacques Heath (Lily May Peel)       |35.0|1     |1       |
|9          |Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  |27.0|3     |1       |
|10         |Nasser, Mrs. Nicholas (Adele Achem)                |14.0|2     |1       |
+-----------+---------------------------------------------------+----+------+--------+
only showing top 5 rows


### 4.3 Pasajeros de primera clase que sobrevivieron

In [24]:
primera_sobrevivientes = spark.sql("""
    SELECT * FROM Titanic
    WHERE Pclass = 1 AND Survived = 1
""")
print("Primera clase que sobrevivieron:", primera_sobrevivientes.count())
primera_sobrevivientes.select("Name", "Sex", "Age", "Fare").show(5, truncate=False)

Primera clase que sobrevivieron: 136
+---------------------------------------------------+------+----+--------+
|Name                                               |Sex   |Age |Fare    |
+---------------------------------------------------+------+----+--------+
|Cumings, Mrs. John Bradley (Florence Briggs Thayer)|female|38.0|71.2833 |
|Futrelle, Mrs. Jacques Heath (Lily May Peel)       |female|35.0|53.1    |
|Bonnell, Miss. Elizabeth                           |female|58.0|26.55   |
|Sloper, Mr. William Thompson                       |male  |28.0|35.5    |
|Spencer, Mrs. William Augustus (Marie Eugenie)     |female|NULL|146.5208|
+---------------------------------------------------+------+----+--------+
only showing top 5 rows


### 4.4 DataFrame sin las columnas Ticket, Fare, Cabin y Embarked

In [25]:
df_reducido = spark.sql("""
    SELECT PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch
    FROM Titanic
""")
print("Columnas originales:", len(df.columns), "→ reducidas:", len(df_reducido.columns))
print(df_reducido.columns)
df_reducido.show(5, truncate=False)

Columnas originales: 12 → reducidas: 8
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch']
+-----------+--------+------+---------------------------------------------------+------+----+-----+-----+
|PassengerId|Survived|Pclass|Name                                               |Sex   |Age |SibSp|Parch|
+-----------+--------+------+---------------------------------------------------+------+----+-----+-----+
|1          |0       |3     |Braund, Mr. Owen Harris                            |male  |22.0|1    |0    |
|2          |1       |1     |Cumings, Mrs. John Bradley (Florence Briggs Thayer)|female|38.0|1    |0    |
|3          |1       |3     |Heikkinen, Miss. Laina                             |female|26.0|0    |0    |
|4          |1       |1     |Futrelle, Mrs. Jacques Heath (Lily May Peel)       |female|35.0|1    |0    |
|5          |0       |3     |Allen, Mr. William Henry                           |male  |35.0|0    |0    |
+-----------+--------+------+------

### 4.5 Pasajeros ordenados por edad, de mayor a menor

`NULLS LAST` deja los pasajeros sin edad al final. Si no lo pongo, Spark los pone primero
y parecería que los más viejos son los que no tienen dato.

In [26]:
spark.sql("""
    SELECT PassengerId, Name, Sex, Age, Pclass, Survived
    FROM Titanic
    ORDER BY Age DESC NULLS LAST
""").show(10, truncate=False)

+-----------+------------------------------------+----+----+------+--------+
|PassengerId|Name                                |Sex |Age |Pclass|Survived|
+-----------+------------------------------------+----+----+------+--------+
|631        |Barkworth, Mr. Algernon Henry Wilson|male|80.0|1     |1       |
|852        |Svensson, Mr. Johan                 |male|74.0|3     |0       |
|97         |Goldschmidt, Mr. George B           |male|71.0|1     |0       |
|494        |Artagaveytia, Mr. Ramon             |male|71.0|1     |0       |
|117        |Connors, Mr. Patrick                |male|70.5|3     |0       |
|673        |Mitchell, Mr. Henry Michael         |male|70.0|2     |0       |
|746        |Crosby, Capt. Edward Gifford        |male|70.0|1     |0       |
|34         |Wheadon, Mr. Edward H               |male|66.0|2     |0       |
|55         |Ostby, Mr. Engelhart Cornelius      |male|65.0|1     |0       |
|281        |Duane, Mr. Frank                    |male|65.0|3     |0       |

### 4.6 Los 10 pasajeros que pagaron las tarifas más altas

In [27]:
spark.sql("""
    SELECT PassengerId, Name, Pclass, Sex, Age, Fare, Survived
    FROM Titanic
    ORDER BY Fare DESC
    LIMIT 10
""").show(truncate=False)

+-----------+-----------------------------------------+------+------+----+--------+--------+
|PassengerId|Name                                     |Pclass|Sex   |Age |Fare    |Survived|
+-----------+-----------------------------------------+------+------+----+--------+--------+
|680        |Cardeza, Mr. Thomas Drake Martinez       |1     |male  |36.0|512.3292|1       |
|259        |Ward, Miss. Anna                         |1     |female|35.0|512.3292|1       |
|738        |Lesurer, Mr. Gustave J                   |1     |male  |35.0|512.3292|1       |
|89         |Fortune, Miss. Mabel Helen               |1     |female|23.0|263.0   |1       |
|28         |Fortune, Mr. Charles Alexander           |1     |male  |19.0|263.0   |0       |
|342        |Fortune, Miss. Alice Elizabeth           |1     |female|24.0|263.0   |1       |
|439        |Fortune, Mr. Mark                        |1     |male  |64.0|263.0   |0       |
|312        |Ryerson, Miss. Emily Borie               |1     |female|1

### 4.7 Columna `Grupo_Edad` con CASE WHEN

El orden de los WHEN importa: SQL evalúa de arriba hacia abajo y se queda con el primero
que se cumple. Por eso el `IS NULL` va primero — si lo dejara al final, nunca se alcanzaría,
porque una comparación con NULL no da verdadero ni falso.

In [28]:
df_edad = spark.sql("""
    SELECT *,
        CASE
            WHEN Age IS NULL              THEN 'Desconocido'
            WHEN Age < 12                 THEN 'Niño'
            WHEN Age BETWEEN 12 AND 17    THEN 'Adolescente'
            WHEN Age BETWEEN 18 AND 59    THEN 'Adulto'
            WHEN Age >= 60                THEN 'Adulto Mayor'
        END AS Grupo_Edad
    FROM Titanic
""")

# Registro esta tabla como vista nueva para poder usar Grupo_Edad en la consulta 4.8
df_edad.createOrReplaceTempView("Titanic_Edad")

df_edad.select("Name", "Age", "Grupo_Edad").show(10, truncate=False)

+---------------------------------------------------+----+-----------+
|Name                                               |Age |Grupo_Edad |
+---------------------------------------------------+----+-----------+
|Braund, Mr. Owen Harris                            |22.0|Adulto     |
|Cumings, Mrs. John Bradley (Florence Briggs Thayer)|38.0|Adulto     |
|Heikkinen, Miss. Laina                             |26.0|Adulto     |
|Futrelle, Mrs. Jacques Heath (Lily May Peel)       |35.0|Adulto     |
|Allen, Mr. William Henry                           |35.0|Adulto     |
|Moran, Mr. James                                   |NULL|Desconocido|
|McCarthy, Mr. Timothy J                            |54.0|Adulto     |
|Palsson, Master. Gosta Leonard                     |2.0 |Niño       |
|Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  |27.0|Adulto     |
|Nasser, Mrs. Nicholas (Adele Achem)                |14.0|Adolescente|
+---------------------------------------------------+----+-----------+
only s

### 4.8 Tasa de supervivencia por grupo de edad

In [29]:
spark.sql("""
    SELECT
        Grupo_Edad,
        SUM(Survived)                              AS Sobrevivientes,
        COUNT(*)                                   AS Total,
        ROUND(100.0 * SUM(Survived) / COUNT(*), 2) AS Pct_Supervivencia
    FROM Titanic_Edad
    GROUP BY Grupo_Edad
    ORDER BY Pct_Supervivencia DESC
""").show()

+------------+--------------+-----+-----------------+
|  Grupo_Edad|Sobrevivientes|Total|Pct_Supervivencia|
+------------+--------------+-----+-----------------+
|        Niño|            39|   68|            57.35|
| Adolescente|            22|   45|            48.89|
|      Adulto|           222|  575|            38.61|
| Desconocido|            52|  177|            29.38|
|Adulto Mayor|             7|   26|            26.92|
+------------+--------------+-----+-----------------+



### 4.9 Tarifa total recaudada por clase

In [30]:
spark.sql("""
    SELECT
        Pclass,
        ROUND(SUM(Fare), 2)  AS Recaudacion_Total,
        ROUND(AVG(Fare), 2)  AS Tarifa_Promedio,
        COUNT(*)             AS Pasajeros
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Recaudacion_Total DESC
""").show()

+------+-----------------+---------------+---------+
|Pclass|Recaudacion_Total|Tarifa_Promedio|Pasajeros|
+------+-----------------+---------------+---------+
|     1|         18177.41|          84.15|      216|
|     3|           6714.7|          13.68|      491|
|     2|          3801.84|          20.66|      184|
+------+-----------------+---------------+---------+



### 4.10 Clases donde la edad promedio de los sobrevivientes supera los 30 años

Acá se ve la diferencia entre `WHERE` y `HAVING`:
- `WHERE Survived = 1` filtra **filas antes** de agrupar (me quedo solo con sobrevivientes).
- `HAVING AVG(Age) > 30` filtra **grupos después** de agrupar (descarta clases completas).

In [31]:
spark.sql("""
    SELECT
        Pclass,
        ROUND(AVG(Age), 2) AS Edad_Promedio_Sobrevivientes,
        COUNT(*)           AS Sobrevivientes
    FROM Titanic
    WHERE Survived = 1
    GROUP BY Pclass
    HAVING AVG(Age) > 30
    ORDER BY Edad_Promedio_Sobrevivientes DESC
""").show()

+------+----------------------------+--------------+
|Pclass|Edad_Promedio_Sobrevivientes|Sobrevivientes|
+------+----------------------------+--------------+
|     1|                       35.37|           136|
+------+----------------------------+--------------+



In [32]:
# Comparación: así se ven TODAS las clases, para dejar claro cuáles descartó el HAVING
spark.sql("""
    SELECT Pclass, ROUND(AVG(Age), 2) AS Edad_Promedio_Sobrevivientes
    FROM Titanic
    WHERE Survived = 1
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+----------------------------+
|Pclass|Edad_Promedio_Sobrevivientes|
+------+----------------------------+
|     1|                       35.37|
|     2|                        25.9|
|     3|                       20.65|
+------+----------------------------+



---
## PARTE 5 · Limpieza de datos y transformaciones

### 5.1 ¿Cuántos nulos hay en Age?

In [33]:
from pyspark.sql import functions as F

nulos_age = df.filter(F.col("Age").isNull()).count()
print(f"Nulos en Age: {nulos_age} de {total} ({round(100*nulos_age/total, 2)} %)")

# Reviso también el resto de las columnas, para saber con qué me estoy enfrentando
print("\nNulos por columna:")
for c in df.columns:
    n = df.filter(F.col(c).isNull()).count()
    if n > 0:
        print(f"  {c:<12} {n:>4}  ({round(100*n/total,2)} %)")

Nulos en Age: 177 de 891 (19.87 %)

Nulos por columna:
  Age           177  (19.87 %)
  Cabin         687  (77.1 %)
  Embarked        2  (0.22 %)


### 5.2 Reemplazar los nulos de Age por la edad promedio

La guía lo pide explícitamente, así que lo hago. Pero dejo anotado lo que observé:

Rellenar con el promedio **mantiene el promedio igual y achica la desviación estándar**,
porque estoy agregando 177 valores idénticos en el centro de la distribución. Los datos
quedan pareciendo más homogéneos de lo que realmente son.

Por eso creo un **DataFrame nuevo** (`df_sin_nulos`) en vez de pisar el original: así puedo
comparar los dos y mostrar el efecto.

In [34]:
# Calculo el promedio una sola vez y lo guardo en una variable de Python
edad_promedio = df.select(F.round(F.avg("Age"), 2)).first()[0]
print("Edad promedio de los pasajeros con dato:", edad_promedio)

# fillna crea un DataFrame nuevo; el original no se toca (los DataFrames son inmutables)
df_sin_nulos = df.fillna({"Age": edad_promedio})

print("Nulos en Age — original:   ", df.filter(F.col("Age").isNull()).count())
print("Nulos en Age — df_sin_nulos:", df_sin_nulos.filter(F.col("Age").isNull()).count())

Edad promedio de los pasajeros con dato: 29.7
Nulos en Age — original:    177
Nulos en Age — df_sin_nulos: 0


In [35]:
# El efecto de rellenar: mismo promedio, menos dispersión
df.select(F.round(F.avg("Age"),2).alias("promedio"),
          F.round(F.stddev("Age"),2).alias("desv_estandar"),
          F.count("Age").alias("con_dato")).show()

df_sin_nulos.select(F.round(F.avg("Age"),2).alias("promedio"),
                    F.round(F.stddev("Age"),2).alias("desv_estandar"),
                    F.count("Age").alias("con_dato")).show()

+--------+-------------+--------+
|promedio|desv_estandar|con_dato|
+--------+-------------+--------+
|    29.7|        14.53|     714|
+--------+-------------+--------+

+--------+-------------+--------+
|promedio|desv_estandar|con_dato|
+--------+-------------+--------+
|    29.7|         13.0|     891|
+--------+-------------+--------+



La desviación estándar baja. El promedio no se mueve. Eso confirma lo que decía arriba:
el relleno no agrega información, solo tapa el hueco.

### 5.3 y 5.4 · Columnas `Descripcion_Sexo` y `Clase_Descripcion`

In [36]:
df_final = (df_sin_nulos
    .withColumn("Descripcion_Sexo",
        F.when(F.col("Sex") == "male",   "Masculino")
         .when(F.col("Sex") == "female", "Femenino")
         .otherwise("Sin dato"))
    .withColumn("Clase_Descripcion",
        F.when(F.col("Pclass") == 1, "Primera Clase")
         .when(F.col("Pclass") == 2, "Segunda Clase")
         .when(F.col("Pclass") == 3, "Tercera Clase")
         .otherwise("Sin dato"))
    .withColumn("Familiares", F.col("SibSp") + F.col("Parch")))

df_final.select("Name", "Sex", "Descripcion_Sexo",
                "Pclass", "Clase_Descripcion", "Familiares").show(10, truncate=False)

+---------------------------------------------------+------+----------------+------+-----------------+----------+
|Name                                               |Sex   |Descripcion_Sexo|Pclass|Clase_Descripcion|Familiares|
+---------------------------------------------------+------+----------------+------+-----------------+----------+
|Braund, Mr. Owen Harris                            |male  |Masculino       |3     |Tercera Clase    |1         |
|Cumings, Mrs. John Bradley (Florence Briggs Thayer)|female|Femenino        |1     |Primera Clase    |1         |
|Heikkinen, Miss. Laina                             |female|Femenino        |3     |Tercera Clase    |0         |
|Futrelle, Mrs. Jacques Heath (Lily May Peel)       |female|Femenino        |1     |Primera Clase    |1         |
|Allen, Mr. William Henry                           |male  |Masculino       |3     |Tercera Clase    |0         |
|Moran, Mr. James                                   |male  |Masculino       |3     |Terc

In [37]:
# Registro la tabla final limpia para usarla en la Parte 6
df_final.createOrReplaceTempView("Titanic_Limpio")
spark.sql("SHOW TABLES").show()

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|         |       titanic|       true|
|         |  titanic_edad|       true|
|         |titanic_limpio|       true|
+---------+--------------+-----------+



---
## PARTE 6 · Análisis multidimensional con Spark SQL

### 6.1 Edad promedio de sobrevivientes vs. no sobrevivientes, por sexo

In [38]:
# Uso la tabla original (Titanic), no la limpia, porque acá quiero la edad REAL.
# Si usara Titanic_Limpio, los 177 rellenos arrastrarían los promedios hacia el centro.
spark.sql("""
    SELECT
        Sex,
        ROUND(AVG(CASE WHEN Survived = 1 THEN Age END), 2) AS Edad_Sobrevivientes,
        ROUND(AVG(CASE WHEN Survived = 0 THEN Age END), 2) AS Edad_No_Sobrevivientes,
        SUM(Survived)                                      AS Sobrevivientes,
        COUNT(*)                                           AS Total
    FROM Titanic
    GROUP BY Sex
    ORDER BY Sex
""").show()

+------+-------------------+----------------------+--------------+-----+
|   Sex|Edad_Sobrevivientes|Edad_No_Sobrevivientes|Sobrevivientes|Total|
+------+-------------------+----------------------+--------------+-----+
|female|              28.85|                 25.05|           233|  314|
|  male|              27.28|                 31.62|           109|  577|
+------+-------------------+----------------------+--------------+-----+



### 6.2 Tarifa promedio de sobrevivientes vs. no sobrevivientes, por clase

In [39]:
spark.sql("""
    SELECT
        Pclass,
        ROUND(AVG(CASE WHEN Survived = 1 THEN Fare END), 2) AS Tarifa_Sobrevivientes,
        ROUND(AVG(CASE WHEN Survived = 0 THEN Fare END), 2) AS Tarifa_No_Sobrevivientes,
        ROUND(AVG(Fare), 2)                                 AS Tarifa_Promedio_Clase
    FROM Titanic
    GROUP BY Pclass
    ORDER BY Pclass
""").show()

+------+---------------------+------------------------+---------------------+
|Pclass|Tarifa_Sobrevivientes|Tarifa_No_Sobrevivientes|Tarifa_Promedio_Clase|
+------+---------------------+------------------------+---------------------+
|     1|                95.61|                   64.68|                84.15|
|     2|                22.06|                   19.41|                20.66|
|     3|                13.69|                   13.67|                13.68|
+------+---------------------+------------------------+---------------------+



### Extra · Cruce de sexo y clase

No lo pide la guía, pero es la consulta que resume el caso completo.

In [40]:
spark.sql("""
    SELECT
        Clase_Descripcion,
        Descripcion_Sexo,
        SUM(Survived)                              AS Sobrevivientes,
        COUNT(*)                                   AS Total,
        ROUND(100.0 * SUM(Survived) / COUNT(*), 2) AS Pct_Supervivencia
    FROM Titanic_Limpio
    GROUP BY Clase_Descripcion, Descripcion_Sexo
    ORDER BY Pct_Supervivencia DESC
""").show()

+-----------------+----------------+--------------+-----+-----------------+
|Clase_Descripcion|Descripcion_Sexo|Sobrevivientes|Total|Pct_Supervivencia|
+-----------------+----------------+--------------+-----+-----------------+
|    Primera Clase|        Femenino|            91|   94|            96.81|
|    Segunda Clase|        Femenino|            70|   76|            92.11|
|    Tercera Clase|        Femenino|            72|  144|            50.00|
|    Primera Clase|       Masculino|            45|  122|            36.89|
|    Segunda Clase|       Masculino|            17|  108|            15.74|
|    Tercera Clase|       Masculino|            47|  347|            13.54|
+-----------------+----------------+--------------+-----+-----------------+



---
## Cierre

Resumen de lo que arrojaron las consultas, con los números de las salidas de arriba.

In [41]:
resumen = spark.sql("""
    SELECT 'Total de pasajeros' AS Indicador, CAST(COUNT(*) AS STRING) AS Valor FROM Titanic
    UNION ALL
    SELECT 'Sobrevivieron', CAST(SUM(Survived) AS STRING) FROM Titanic
    UNION ALL
    SELECT 'Supervivencia general (%)',
           CAST(ROUND(100.0*SUM(Survived)/COUNT(*),2) AS STRING) FROM Titanic
    UNION ALL
    SELECT 'Supervivencia mujeres (%)',
           CAST(ROUND(100.0*SUM(Survived)/COUNT(*),2) AS STRING) FROM Titanic WHERE Sex='female'
    UNION ALL
    SELECT 'Supervivencia hombres (%)',
           CAST(ROUND(100.0*SUM(Survived)/COUNT(*),2) AS STRING) FROM Titanic WHERE Sex='male'
    UNION ALL
    SELECT 'Supervivencia 1a clase (%)',
           CAST(ROUND(100.0*SUM(Survived)/COUNT(*),2) AS STRING) FROM Titanic WHERE Pclass=1
    UNION ALL
    SELECT 'Supervivencia 3a clase (%)',
           CAST(ROUND(100.0*SUM(Survived)/COUNT(*),2) AS STRING) FROM Titanic WHERE Pclass=3
    UNION ALL
    SELECT 'Nulos en Age', CAST(COUNT(*) AS STRING) FROM Titanic WHERE Age IS NULL
""")
resumen.show(truncate=False)

+--------------------------+-----+
|Indicador                 |Valor|
+--------------------------+-----+
|Total de pasajeros        |891  |
|Sobrevivieron             |342  |
|Supervivencia general (%) |38.38|
|Supervivencia mujeres (%) |74.20|
|Supervivencia hombres (%) |18.89|
|Supervivencia 1a clase (%)|62.96|
|Supervivencia 3a clase (%)|24.24|
|Nulos en Age              |177  |
+--------------------------+-----+



**Lo que muestran los números:**

1. El sexo pesa más que cualquier otra variable. La diferencia entre mujeres y hombres es la
   brecha más grande de todo el análisis.
2. La clase también separa con claridad: primera contra tercera es una diferencia enorme.
3. Las dos cosas se refuerzan. El cruce de sexo y clase deja los extremos a la vista.
4. 177 pasajeros no tienen edad registrada. Eso es un 19,87 % del dataset — no es un detalle,
   y por eso dejé el relleno en un DataFrame aparte en vez de pisar el original.

**Sobre Spark SQL:** las 28 consultas salieron escritas en SQL. Spark las convierte al mismo
plan de ejecución que si las hubiera escrito con métodos de DataFrame (`.groupBy().agg()`),
así que el rendimiento es idéntico. Elegí SQL porque las agregaciones con `CASE WHEN` adentro
se leen mejor.

In [42]:
# Libero los recursos del clúster local
spark.stop()
print("Sesión de Spark cerrada")

Sesión de Spark cerrada
